# Automations: Scheduling Agent Queries with MemoRizz

Automations let you schedule your MemoRizz agents to run queries on a recurring basis — daily briefings, hourly monitoring, one-off reports — without a human in the loop.

**What you'll learn in this notebook:**

1. How to create automations using the **SDK** (Python code)
2. The three **schedule types**: cron, interval, and one-shot
3. Real-world **use cases**: daily briefings, monitoring, report generation
4. Managing automations: pause, resume, trigger, delete
5. **Delivery channels**: in-chat vs WhatsApp
6. Alternative creation methods: Web UI and agent conversation

> **Prerequisites:** Oracle Database running with MemoRizz schema set up. If you haven't done this yet, see the `single_agent/memagent_local_oracle.ipynb` notebook first.

---
# Setup

Install required packages and configure environment variables.

In [ ]:
# Install memorizz and required dependencies
%pip install -qU memorizz
%pip install -qU oracledb
%pip install -qU openai
%pip install -qU python-dotenv

print("All packages installed!")

In [ ]:
import os

# Oracle connection
ORACLE_USER = os.getenv("ORACLE_USER", "memorizz_user")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD", "SecurePass123!")
ORACLE_DSN = os.getenv("ORACLE_DSN", "localhost:1521/FREEPDB1")

os.environ["ORACLE_USER"] = ORACLE_USER
os.environ["ORACLE_PASSWORD"] = ORACLE_PASSWORD
os.environ["ORACLE_DSN"] = ORACLE_DSN
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_PROVIDER"] = "openai"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_MODEL"] = "text-embedding-3-small"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS"] = "256"

In [ ]:
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

print("OpenAI API key configured.")

In [ ]:
import logging

os.environ["MEMORIZZ_LOG_LEVEL"] = "INFO"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    force=True,
)

---
# Part 1: Create an Agent

Before we can schedule automations, we need an agent. Let's create a finance assistant that we'll schedule to run daily market briefings, hourly monitoring checks, and on-demand reports.

In [ ]:
from memorizz.memory_provider.oracle import OracleConfig, OracleProvider

oracle_config = OracleConfig(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    schema=ORACLE_USER,
    lazy_vector_indexes=False,
    embedding_provider="openai",
    embedding_config={
        "model": "text-embedding-3-small",
        "dimensions": 256,
        "api_key": os.getenv("OPENAI_API_KEY"),
    },
)

oracle_memory_provider = OracleProvider(oracle_config)
print("Oracle provider initialized!")

In [ ]:
from memorizz.memagent.builders import MemAgentBuilder

agent = (
    MemAgentBuilder()
    .with_instruction(
        "You are a finance assistant that provides market analysis, "
        "portfolio insights, and economic briefings. When given a date, "
        "provide analysis relevant to that date. Be concise and actionable."
    )
    .with_memory_provider(oracle_memory_provider)
    .with_llm_config(
        {
            "provider": "openai",
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY"),
        }
    )
    .with_automations_enabled(True)
    .with_default_timezone("America/New_York")
    .build()
)

agent.save()
print(f"Agent created: {agent.agent_id}")
print(f"Automations available: {agent.has_automations()}")

Two important builder calls here:

- **`.with_automations_enabled(True)`** — Enables the automation system for this agent. This is actually the default, but being explicit makes the intent clear.
- **`.with_default_timezone("America/New_York")`** — Sets a default timezone so you don't have to pass it every time you create an automation.

You can verify automation support with `agent.has_automations()`. This returns `True` only when:
1. The agent is connected to an Oracle memory provider
2. `automations_enabled` is `True`
3. The automation tables exist in the database

## Adding Automations to an Existing Agent

If you already have a saved agent, you don't need to create a new one. Use `MemAgent.load()` to load it and start creating automations immediately — as long as the agent was built with an Oracle provider.

In [ ]:
# Loading an existing agent and using automations
# (Uncomment and replace with your agent ID)

# from memorizz.memagent import MemAgent
#
# existing_agent = MemAgent.load(
#     "your-agent-id-here",
#     memory_provider=oracle_memory_provider,
# )
# print(f"Loaded agent: {existing_agent.agent_id}")
# print(f"Automations available: {existing_agent.has_automations()}")
#
# # Now you can create automations on the loaded agent
# job = existing_agent.create_automation(
#     name="Loaded Agent Briefing",
#     schedule_type="cron",
#     cron_expr="0 9 * * *",
#     timezone="UTC",
#     query_template="Good morning! Summarize today's headlines for {today_iso}.",
# )
# print(f"Created automation: {job.name}")

print("MemAgent.load() example (commented out — replace with your agent ID)")

## Common Errors and How to Handle Them

Before diving into creating automations, here's what happens when things go wrong — and how to handle it gracefully.

In [ ]:
# Error 1: Automations not available (no Oracle provider)
# This happens when you use a non-Oracle provider (e.g. FileSystem, MongoDB)

from memorizz.memagent import MemAgent

no_oracle_agent = MemAgent()  # No memory provider
print(f"has_automations(): {no_oracle_agent.has_automations()}")

try:
    no_oracle_agent.create_automation(
        name="This will fail",
        schedule_type="interval",
        query_template="Hello",
        interval_seconds=60,
        timezone="UTC",
    )
except ValueError as e:
    print(f"ValueError: {e}")
    # → "Automations are not available. Requires an Oracle memory provider
    #    with automations_enabled=True."

In [ ]:
# Error 2: Missing timezone
# If the agent has no default_timezone and you don't pass one explicitly,
# create_automation raises a clear error telling you how to fix it.

# To demonstrate, we'd need an agent without default_timezone set.
# Our agent above has default_timezone="America/New_York", so this would work
# even without an explicit timezone parameter. But if you create an agent
# without .with_default_timezone(), you'll see:
#
# try:
#     agent_no_tz.create_automation(
#         name="Test",
#         schedule_type="interval",
#         query_template="Hello",
#         interval_seconds=60,
#     )
# except ValueError as e:
#     print(f"ValueError: {e}")
#     # → "timezone is required. Pass it explicitly, set default_timezone
#     #    on the agent, or set the MEMORIZZ_DEFAULT_TIMEZONE env var."

# Best practice: always check has_automations() before calling SDK methods
if agent.has_automations():
    print("Automations are available — safe to create jobs")
else:
    print("Automations not available — check your provider and configuration")

---
# Part 2: Creating Automations with the SDK

There are **three ways** to create automations in MemoRizz:

| Method | Best for |
|--------|----------|
| **SDK** (Python code) | Scripts, CI/CD, programmatic setup |
| **Web UI** (`/automations`) | Visual configuration, quick setup |
| **Agent Conversation** | Natural language, asking the agent directly |

This notebook focuses on the **SDK approach**. We'll cover all three schedule types with real-world use cases.

## Use Case 1: Daily Market Briefing (Cron Schedule)

The most common automation pattern: run a query at a specific time every day. We'll use a **cron schedule** — the same syntax used by Unix cron jobs.

**Cron format:** `minute hour day_of_month month day_of_week`

| Expression | Meaning |
|------------|----------|
| `0 9 * * *` | Every day at 9:00 AM |
| `0 9 * * 1-5` | Weekdays at 9:00 AM |
| `30 */2 * * *` | Every 2 hours at :30 |
| `0 0 1 * *` | First of every month at midnight |

In [ ]:
daily_briefing = agent.create_automation(
    name="Morning Market Briefing",
    schedule_type="cron",
    cron_expr="0 8 * * 1-5",  # Weekdays at 8:00 AM
    query_template=(
        "It's {today_iso} in the {timezone} timezone. "
        "Give me a morning market briefing covering:\n"
        "1. Pre-market futures and overnight moves\n"
        "2. Key economic data releases scheduled for today\n"
        "3. Earnings reports expected today\n"
        "4. Any breaking news that could move markets\n"
        "Keep it concise — bullet points preferred."
    ),
)

print(f"Created: {daily_briefing.name}")
print(f"Job ID: {daily_briefing.job_id}")
print(f"Schedule: {daily_briefing.cron_expr} ({daily_briefing.timezone})")
print(f"Next run: {daily_briefing.next_run_at}")

That's it. The `create_automation` method handles all the boilerplate:

- Generates a unique job ID and memory ID
- Validates the cron expression and timezone
- Computes `next_run_at` based on the cron schedule
- Persists the job to Oracle

Notice the **query template** uses placeholders:
- `{today_iso}` — Today's date in the job's timezone (e.g. `2025-01-15`)
- `{timezone}` — The configured timezone name
- `{scheduled_for_iso}` — The exact scheduled execution time
- `{now_utc_iso}` — Current UTC time

These are rendered at execution time, so each run gets the correct date.

## Use Case 2: Monitoring Dashboard (Interval Schedule)

For regular polling at fixed intervals — like checking a system every 30 minutes — use an **interval schedule**. You specify the number of seconds between runs.

In [ ]:
monitor = agent.create_automation(
    name="Portfolio Risk Monitor",
    schedule_type="interval",
    interval_seconds=1800,  # Every 30 minutes
    query_template=(
        "Run a risk check on my portfolio as of {now_utc_iso}.\n"
        "Flag any positions that have moved more than 3%% since market open.\n"
        "If nothing significant, reply with 'All positions within normal range.'"
    ),
)

print(f"Created: {monitor.name}")
print(f"Interval: every {monitor.interval_seconds} seconds")
print(f"Next run: {monitor.next_run_at}")

**When to use interval vs cron:**

- **Cron** is best when you need a specific time of day: "every morning at 8 AM", "first Monday of the month"
- **Interval** is best for regular polling: "every 30 minutes", "every 2 hours"

With intervals, the clock starts from when the job is created. So `interval_seconds=1800` means the first run is 30 minutes from now, then every 30 minutes after that.

## Use Case 3: One-Time Report (One-Shot Schedule)

Sometimes you just need to schedule a single execution — maybe a report that should run at 6 PM today, or a migration task. Use **one_shot** for this. After it runs once, the job automatically disables itself.

In [ ]:
report = agent.create_automation(
    name="Q4 Performance Report",
    schedule_type="one_shot",
    query_template=(
        "Generate a comprehensive Q4 performance report covering:\n"
        "1. Total portfolio return vs S&P 500 benchmark\n"
        "2. Best and worst performing positions\n"
        "3. Sector allocation changes during the quarter\n"
        "4. Risk-adjusted metrics (Sharpe ratio, max drawdown)\n"
        "5. Recommendations for Q1 rebalancing\n\n"
        "Format as a professional report with sections and bullet points."
    ),
)

print(f"Created: {report.name}")
print(f"Schedule: {report.schedule_type} (runs once, then disables)")
print(f"Next run: {report.next_run_at}")

---
# Part 3: Managing Automations

Once automations are created, you have full lifecycle control through the SDK.

## Listing All Automations

See all automation jobs for this agent. You can filter by enabled/disabled state.

In [ ]:
# List all automations for this agent
all_jobs = agent.list_automations()
print(f"Total automations: {len(all_jobs)}\n")

for job in all_jobs:
    print(f"  {job.name}")
    print(f"    ID: {job.job_id}")
    print(f"    Schedule: {job.schedule_type}", end="")
    if job.cron_expr:
        print(f" ({job.cron_expr})")
    elif job.interval_seconds:
        print(f" (every {job.interval_seconds}s)")
    else:
        print()
    print(f"    Enabled: {job.enabled}")
    print(f"    Next run: {job.next_run_at}")
    print()

In [ ]:
# Filter: only enabled jobs
enabled_jobs = agent.list_automations(enabled=True)
print(f"Enabled automations: {len(enabled_jobs)}")

## Pausing and Resuming

Pause a job to temporarily stop it from running. The schedule is preserved — when you resume, it picks up from where it left off.

In [ ]:
# Pause the monitor
paused = agent.pause_automation(monitor.job_id)
print(f"{paused.name}: enabled={paused.enabled}")

# Verify it's excluded from enabled list
enabled_jobs = agent.list_automations(enabled=True)
print(f"Enabled automations: {len(enabled_jobs)}")

In [ ]:
# Resume it
resumed = agent.resume_automation(monitor.job_id)
print(f"{resumed.name}: enabled={resumed.enabled}")

## Triggering an Immediate Run

Don't want to wait for the next scheduled time? Use `trigger_automation` to run a job immediately. This sets `next_run_at` to now — the worker will pick it up on its next poll cycle (typically within 5 seconds).

In [ ]:
triggered = agent.trigger_automation(daily_briefing.job_id)
print(f"Triggered: {triggered.name}")
print(f"Next run at: {triggered.next_run_at} (should be ~now)")

> **Note:** `trigger_automation` doesn't execute the job itself — it tells the worker to pick it up ASAP. You need a running worker for the job to actually execute. See Part 6 below.

## Viewing Run History

After a job executes, you can inspect its run history — status, timing, errors, and the response payload.

In [ ]:
runs = agent.list_automation_runs(daily_briefing.job_id, limit=5)

if runs:
    for run in runs:
        print(f"  Run: {run.run_id[:8]}...")
        print(f"    Status: {run.status}")
        print(f"    Scheduled for: {run.scheduled_for}")
        print(f"    Started: {run.started_at}")
        print(f"    Finished: {run.finished_at}")
        if run.error:
            print(f"    Error: {run.error}")
        if run.result_payload and run.result_payload.get("response"):
            snippet = run.result_payload["response"][:200]
            print(f"    Response: {snippet}...")
        print()
else:
    print("No runs yet. Start a worker and trigger the job, or wait for the scheduled time.")

## Looking Up and Deleting a Job

In [ ]:
# Look up a specific job by ID
found = agent.get_automation(report.job_id)
if found:
    print(f"Found: {found.name} (schedule: {found.schedule_type})")
else:
    print("Job not found")

In [ ]:
# Delete the one-shot report (it's done its job)
deleted = agent.delete_automation(report.job_id)
print(f"Deleted: {deleted}")

# Verify
remaining = agent.list_automations()
print(f"Remaining automations: {len(remaining)}")
for job in remaining:
    print(f"  - {job.name}")

---
# Part 4: Delivery Channels

By default, automation results are stored **in-chat** — they appear in the agent's conversation memory and are visible in the playground's Automations panel.

You can also deliver results via **WhatsApp** using Twilio. This is useful for alerts that need to reach someone's phone immediately.

## In-Chat Delivery (Default)

This is what we've been doing — no extra configuration needed. The agent's response is saved to its conversation memory, and you can view it in:

- The **Playground** (Automations panel)
- The **Traces** page
- Via `agent.list_automation_runs()` in the SDK

## WhatsApp Delivery (via Twilio)

To deliver results to WhatsApp, pass the `whatsapp_to` parameter with a list of phone numbers. You'll also need Twilio credentials as environment variables:

```bash
export TWILIO_ACCOUNT_SID="your_account_sid"
export TWILIO_AUTH_TOKEN="your_auth_token"
export TWILIO_WHATSAPP_FROM="whatsapp:+14155238886"
```

In [ ]:
# Example: Create an automation with WhatsApp delivery
# Uncomment and update the phone numbers to use this

# alert = agent.create_automation(
#     name="Portfolio Alert (WhatsApp)",
#     schedule_type="interval",
#     interval_seconds=3600,  # Every hour
#     query_template=(
#         "Check my portfolio as of {now_utc_iso}. "
#         "If any position is down more than 5%% today, send an alert. "
#         "Otherwise reply with 'All positions stable.'"
#     ),
#     whatsapp_to=["+15551234567", "+15559876543"],
# )
# print(f"Created: {alert.name}")
# print(f"Delivery: {alert.delivery_type}")
# print(f"Recipients: {alert.delivery_config.get('whatsapp_to')}")

print("WhatsApp delivery example (commented out — update phone numbers to use)")

---
# Part 5: Query Template Patterns

The `query_template` is the prompt your agent receives at execution time. Here are some proven patterns for different use cases.

## Pattern 1: Date-Aware Briefings

Use `{today_iso}` and `{timezone}` to give the agent temporal context.

```python
query_template=(
    "Today is {today_iso} ({timezone}). "
    "Summarize the top 5 tech news stories from today."
)
```

## Pattern 2: Monitoring with Thresholds

Tell the agent to only report when something is noteworthy. This keeps noise down.

```python
query_template=(
    "Check system status as of {now_utc_iso}.\n"
    "Only report if any of these conditions are true:\n"
    "- Error rate exceeds 1%%\n"
    "- Response time p99 > 500ms\n"
    "- Any service is down\n"
    "Otherwise reply: 'All systems nominal.'"
)
```

## Pattern 3: Structured Reports

Request specific formatting so the output is consistent across runs.

```python
query_template=(
    "Generate the weekly status report for {today_iso}.\n\n"
    "Use this format:\n"
    "## Completed This Week\n"
    "- [item]\n\n"
    "## In Progress\n"
    "- [item] (expected completion: [date])\n\n"
    "## Blockers\n"
    "- [blocker] (impact: [description])\n\n"
    "## Next Week Priorities\n"
    "- [priority]"
)
```

## Pattern 4: Multi-Step Analysis

If your agent has tools, the automation query can trigger multi-step workflows.

```python
query_template=(
    "Perform the daily portfolio analysis for {today_iso}:\n"
    "1. Fetch current prices for all positions\n"
    "2. Calculate daily P&L for each position\n"
    "3. Check if any position breaches its stop-loss level\n"
    "4. Summarize findings in a table format"
)
```

> **Tip:** Automations run with a special directive that tells the agent not to ask clarifying questions. This means your query template should be self-contained — don't leave ambiguity for the agent to ask about.

---
# Part 6: Running the Worker

Automations don't execute on their own — they need a **worker** process that polls the database for due jobs and runs them.

There are two ways to run the worker:

## Option 1: CLI (Recommended for production)

Run the worker as a standalone process:

```bash
memorizz run-automations --poll-interval 5 --lease-seconds 120 --concurrency 2
```

| Flag | Default | Description |
|------|---------|-------------|
| `--poll-interval` | 5 | Seconds between polling the database for due jobs |
| `--lease-seconds` | 120 | Lock duration (prevents duplicate execution) |
| `--concurrency` | 2 | Max concurrent job executions |

Required environment variables: `ORACLE_USER`, `ORACLE_PASSWORD`, `ORACLE_DSN`.

## Option 2: Embedded in the Web UI

When running the MemoRizz dashboard (`memorizz run-local`), you can enable the embedded worker in **Settings**. This is convenient for development — the worker runs inside the same process as the web UI.

---
# Part 7: Alternative Creation Methods

While this notebook focused on the SDK, here's how to create automations via the other two methods.

## Web UI

1. Open the MemoRizz dashboard: `memorizz run-local`
2. Navigate to an agent's **Playground** page
3. Click **Create Automation** (or go to `/automations/new`)
4. Fill in the form:
   - Select a **Quick Schedule** preset (daily, weekday, hourly, etc.) or configure manually
   - Write the query template
   - Choose a delivery channel (In Chat or WhatsApp)
5. Click **Create Automation**

The UI provides schedule presets with a time picker, so you don't need to write cron expressions by hand.

## Agent Conversation

When `has_automations()` is `True`, the agent can create automations via natural language. Six tool functions are automatically registered:

- `automation_create_job`
- `automation_list_jobs`
- `automation_pause_job`
- `automation_resume_job`
- `automation_delete_job`
- `automation_run_now`

Simply ask the agent in conversation:

In [ ]:
# The agent can create automations through conversation!
# (This uses the agent's built-in automation tools)

response = agent.run(
    "Schedule a daily automation called 'End of Day Summary' "
    "that runs every weekday at 5 PM Eastern time. "
    "The query should ask for a summary of today's market moves "
    "and any after-hours news. Confirm the creation."
)
print(response)

The agent will call `automation_create_job` internally. The tool-based approach includes a **confirmation step** — the agent shows you what it's about to create and waits for you to confirm. This is by design, since LLM-based creation needs a safety net.

---
# Part 8: Cleanup

Let's clean up the automations we created in this notebook.

In [ ]:
# Delete all automations for this agent
all_jobs = agent.list_automations()
for job in all_jobs:
    agent.delete_automation(job.job_id)
    print(f"Deleted: {job.name}")

remaining = agent.list_automations()
print(f"\nRemaining automations: {len(remaining)}")

---
# Summary

In this notebook, we covered:

| Topic | Key takeaway |
|-------|--------------|
| **Cron schedules** | For specific times: `0 8 * * 1-5` = weekdays at 8 AM |
| **Interval schedules** | For regular polling: `interval_seconds=1800` = every 30 min |
| **One-shot schedules** | Run once, then auto-disable |
| **Query templates** | Use `{today_iso}`, `{timezone}`, `{now_utc_iso}` placeholders |
| **Management** | `list`, `pause`, `resume`, `trigger`, `delete` |
| **Delivery** | In-chat (default) or WhatsApp via Twilio |
| **Three creation methods** | SDK, Web UI, Agent Conversation |

**Next steps:**
- Start a worker: `memorizz run-automations`
- View results in the Web UI: `memorizz run-local` → Agent Playground → Automations panel
- Read the full reference: `src/memorizz/automation/README.md`